# 05 — Equipamientos y análisis de accesibilidad

## Objetivo
Calcular indicadores de accesibilidad a servicios esenciales (salud y educación)
para las manzanas de Apartadó, identificando **desiertos de servicios** — zonas
sin acceso razonable a ninguno de estos equipamientos.

## Fuentes de datos

### Salud — REPS (Registro Especial de Prestadores de Servicios de Salud)
- **Portal SISPRO**: https://www.sispro.gov.co/central-prestadores/Pages/RegistroEspecialPrestadoresSalud.aspx
- **Datos Abiertos**: https://www.datos.gov.co/Salud-y-Protecci-n-Social/Registro-Especial-de-Prestadores-de-Salud-REPS/756x-xf2q
- Descargar para Antioquia y filtrar por municipios de Urabá
- Guardar en: `data/raw/reps/reps_antioquia.csv`

### Educación — SIMAT (Sistema Integrado de Matrícula)
- **Portal MEN**: https://www.mineducacion.gov.co/1759/w3-article-235111.html
- **Datos Abiertos**: https://www.datos.gov.co/Educaci-n/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR-BASICA/ji8i-4gut
- Descargar establecimientos educativos para Antioquia
- Guardar en: `data/raw/simat/simat_establecimientos.csv`

### OSM (OpenStreetMap)
- Hospitales, clínicas y escuelas disponibles vía `osmnx` sin registro previo.
- Fallback automático si REPS/SIMAT no están disponibles.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import contextily as ctx
import warnings
warnings.filterwarnings("ignore")

from src.ingestion.dane_loader import cargar_mgn_manzanas
from src.indicators.base import Indicator, CompositeIndicator
from src.indicators.accesibilidad.areas_verdes import IndicadorAreasVerdes

DIVIPOLA_APARTADO = "05045"
CRS_GEO = "EPSG:4326"
CRS_COL = "EPSG:3116"
DATA_RAW  = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"

# Cargar o generar manzanas dummy Apartadó
mgn_path = DATA_RAW / "dane" / "MGN_MANZANA.shp"
if mgn_path.exists():
    manzanas_all = cargar_mgn_manzanas(mgn_path)
    manzanas_apartado = manzanas_all[
        manzanas_all["cod_manzana"].str.startswith(DIVIPOLA_APARTADO)
    ].copy()
    print(f"Manzanas Apartadó (MGN): {len(manzanas_apartado)}")
else:
    from shapely.geometry import box as shapely_box
    n = 15
    lons = np.linspace(-76.65, -76.61, n)
    lats = np.linspace(7.86, 7.90, n)
    geoms, codes = [], []
    for i in range(len(lons)-1):
        for j in range(len(lats)-1):
            geoms.append(shapely_box(lons[i], lats[j], lons[i+1], lats[j+1]))
            codes.append(f"05045{i:02d}{j:02d}")
    manzanas_apartado = gpd.GeoDataFrame(
        {"cod_manzana": codes, "poblacion": np.random.randint(50, 400, len(codes))},
        geometry=geoms, crs=CRS_GEO
    )
    print(f"Manzanas dummy: {len(manzanas_apartado)}")


In [ ]:
# ── Cargar REPS (establecimientos de salud) ───────────────────────────────
def cargar_reps(data_dir: Path = DATA_RAW / "reps") -> gpd.GeoDataFrame:
    """
    Carga el Registro Especial de Prestadores de Servicios de Salud (REPS).
    Busca: data/raw/reps/reps_antioquia.csv
    Columnas esperadas: CODIGO_HABILITACION, NOMBRE_PRESTADOR, MUNICIPIO_CODIGO,
                        LATITUD, LONGITUD, NIVEL_ATENCION
    Fallback: descarga hospitales y clínicas desde OSM vía osmnx.
    """
    reps_path = data_dir / "reps_antioquia.csv"
    divipolas_uraba = ["05045", "05147", "05120", "05837", "05544", "05665", "05659", "05051"]

    if reps_path.exists():
        df = pd.read_csv(reps_path, dtype={"MUNICIPIO_CODIGO": str}, low_memory=False)
        df = df[df["MUNICIPIO_CODIGO"].isin(divipolas_uraba)].copy()
        df = df.dropna(subset=["LATITUD", "LONGITUD"])
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["LONGITUD"], df["LATITUD"]),
            crs=CRS_GEO
        )
        print(f"[REPS] {len(gdf):,} establecimientos de salud en Urabá")
        return gdf

    # Fallback OSM
    print("[REPS] Archivo no encontrado — descargando desde OSM...")
    try:
        import osmnx as ox
        bbox_ap = manzanas_apartado.total_bounds
        tags = {"amenity": ["hospital", "clinic", "doctors", "health_center", "pharmacy"]}
        salud = ox.features_from_bbox(
            bbox=(bbox_ap[3], bbox_ap[1], bbox_ap[2], bbox_ap[0]),
            tags=tags
        ).to_crs(CRS_GEO)
        salud["NOMBRE_PRESTADOR"] = salud.get("name", pd.Series("", index=salud.index))
        salud["NIVEL_ATENCION"] = salud.get("amenity", pd.Series("", index=salud.index))
        # Convertir a centroides si son polígonos
        salud = salud.copy()
        salud.geometry = salud.geometry.centroid
        print(f"[OSM] {len(salud)} equipamientos de salud")
        return salud[["NOMBRE_PRESTADOR", "NIVEL_ATENCION", "geometry"]]
    except Exception as e:
        print(f"[OSM] Error: {e} — generando datos dummy")
        # Puntos aleatorios en Apartadó
        from shapely.geometry import Point
        bbox = manzanas_apartado.total_bounds
        n_salud = 8
        xs = np.random.uniform(bbox[0], bbox[2], n_salud)
        ys = np.random.uniform(bbox[1], bbox[3], n_salud)
        return gpd.GeoDataFrame(
            {"NOMBRE_PRESTADOR": [f"Centro Salud {i+1}" for i in range(n_salud)],
             "NIVEL_ATENCION": np.random.choice(["I", "II", "III"], n_salud)},
            geometry=[Point(x, y) for x, y in zip(xs, ys)],
            crs=CRS_GEO
        )

salud_gdf = cargar_reps()
print(f"\nTotal establecimientos salud: {len(salud_gdf)}")
if not salud_gdf.empty:
    display(salud_gdf.head(3))


In [ ]:
# ── Cargar SIMAT (establecimientos educativos) ────────────────────────────
def cargar_simat(data_dir: Path = DATA_RAW / "simat") -> gpd.GeoDataFrame:
    """
    Carga el listado de establecimientos educativos del SIMAT (MEN).
    Busca: data/raw/simat/simat_establecimientos.csv
    Columnas esperadas: CODIGO_DANE_ESTABLECIMIENTO, NOMBRE_ESTABLECIMIENTO,
                        MUNICIPIO_CODIGO, LATITUD, LONGITUD, SECTOR (Oficial/Privado)
    Fallback: descarga colegios/escuelas desde OSM vía osmnx.
    """
    simat_path = data_dir / "simat_establecimientos.csv"
    divipolas_uraba = ["05045", "05147", "05120", "05837", "05544", "05665", "05659", "05051"]

    if simat_path.exists():
        df = pd.read_csv(simat_path, dtype={"MUNICIPIO_CODIGO": str}, low_memory=False)
        df = df[df["MUNICIPIO_CODIGO"].isin(divipolas_uraba)].copy()
        df = df.dropna(subset=["LATITUD", "LONGITUD"])
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["LONGITUD"], df["LATITUD"]),
            crs=CRS_GEO
        )
        print(f"[SIMAT] {len(gdf):,} establecimientos educativos en Urabá")
        return gdf

    # Fallback OSM
    print("[SIMAT] Archivo no encontrado — descargando desde OSM...")
    try:
        import osmnx as ox
        bbox_ap = manzanas_apartado.total_bounds
        tags = {"amenity": ["school", "kindergarten", "college", "university"]}
        educ = ox.features_from_bbox(
            bbox=(bbox_ap[3], bbox_ap[1], bbox_ap[2], bbox_ap[0]),
            tags=tags
        ).to_crs(CRS_GEO)
        educ = educ.copy()
        educ.geometry = educ.geometry.centroid
        educ["NOMBRE_ESTABLECIMIENTO"] = educ.get("name", pd.Series("", index=educ.index))
        educ["SECTOR"] = "OSM"
        print(f"[OSM] {len(educ)} equipamientos educativos")
        return educ[["NOMBRE_ESTABLECIMIENTO", "SECTOR", "geometry"]]
    except Exception as e:
        print(f"[OSM] Error: {e} — generando datos dummy")
        from shapely.geometry import Point
        bbox = manzanas_apartado.total_bounds
        n_educ = 12
        xs = np.random.uniform(bbox[0], bbox[2], n_educ)
        ys = np.random.uniform(bbox[1], bbox[3], n_educ)
        return gpd.GeoDataFrame(
            {"NOMBRE_ESTABLECIMIENTO": [f"IE {i+1}" for i in range(n_educ)],
             "SECTOR": np.random.choice(["Oficial", "Privado"], n_educ)},
            geometry=[Point(x, y) for x, y in zip(xs, ys)],
            crs=CRS_GEO
        )

educ_gdf = cargar_simat()
print(f"\nTotal establecimientos educativos: {len(educ_gdf)}")
if not educ_gdf.empty:
    display(educ_gdf.head(3))


In [ ]:
# ── IndicadorSalud: distancia mínima a establecimiento de salud ───────────
class IndicadorSalud(Indicator):
    """
    Indicador de Accesibilidad a Salud (ISAL).
    Calcula distancia peatonal estimada (buffer euclidiana) desde cada
    manzana al establecimiento de salud más cercano, luego normaliza e invierte:
    distancia corta → ISAL alto (mejor).
    """
    name = "ISAL"
    dimension = "accesibilidad"
    unit = "metros al establecimiento de salud más cercano"
    invert = True  # distancia alta = peor

    def __init__(self, manzanas: gpd.GeoDataFrame, equipamientos: gpd.GeoDataFrame):
        super().__init__(manzanas)
        self.equipamientos = equipamientos

    def calculate(self) -> pd.Series:
        manzanas_proj = self.manzanas.to_crs(CRS_COL).copy()
        manzanas_proj["centroide"] = manzanas_proj.geometry.centroid

        if self.equipamientos.empty:
            return pd.Series(9999.0, index=self.manzanas.set_index("cod_manzana").index)

        equip_proj = self.equipamientos.to_crs(CRS_COL).copy()
        equip_pts = equip_proj.geometry.centroid

        distancias = {}
        for _, row in manzanas_proj.iterrows():
            dists = equip_pts.distance(row["centroide"])
            distancias[row["cod_manzana"]] = dists.min()
        return pd.Series(distancias, name=self.name)

isal_ind = IndicadorSalud(manzanas_apartado, salud_gdf)
isal_score = isal_ind.run()
print(f"ISAL calculado: {len(isal_score)} manzanas")
print(isal_score.describe().round(4))

# Unir al GeoDataFrame
manzanas_apartado = manzanas_apartado.merge(
    isal_score.rename("ISAL").reset_index().rename(columns={"index": "cod_manzana"}),
    on="cod_manzana", how="left"
)

# Mapa ISAL
fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_apartado.to_crs(CRS_COL)
manzanas_proj.plot(
    column="ISAL", ax=ax, cmap="RdYlGn", legend=True,
    legend_kwds={"label": "ISAL — Accesibilidad a Salud (0=peor, 1=mejor)", "shrink": 0.7},
    edgecolor="white", linewidth=0.3, alpha=0.85, vmin=0, vmax=1
)
# Superponer equipamientos
if not salud_gdf.empty:
    salud_proj = salud_gdf.to_crs(CRS_COL)
    salud_proj.plot(ax=ax, color="red", marker="+", markersize=80,
                    label="Establecimientos salud", zorder=5)
try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass
ax.set_title("Accesibilidad a Salud (ISAL) — Apartadó", fontsize=14, fontweight="bold")
ax.legend(loc="lower left")
ax.set_axis_off()
plt.tight_layout()
(ROOT / "data" / "outputs").mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT / "data" / "outputs" / "isal_apartado.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── IndicadorEducacion: accesibilidad a establecimientos educativos ────────
class IndicadorEducacion(Indicator):
    """
    Indicador de Accesibilidad a Educación (ISE).
    Igual lógica que ISAL pero usando establecimientos educativos del SIMAT.
    """
    name = "ISE"
    dimension = "accesibilidad"
    unit = "metros al establecimiento educativo más cercano"
    invert = True

    def __init__(self, manzanas: gpd.GeoDataFrame, equipamientos: gpd.GeoDataFrame):
        super().__init__(manzanas)
        self.equipamientos = equipamientos

    def calculate(self) -> pd.Series:
        manzanas_proj = self.manzanas.to_crs(CRS_COL).copy()
        manzanas_proj["centroide"] = manzanas_proj.geometry.centroid

        if self.equipamientos.empty:
            return pd.Series(9999.0, index=self.manzanas.set_index("cod_manzana").index)

        equip_proj = self.equipamientos.to_crs(CRS_COL).copy()
        equip_pts = equip_proj.geometry.centroid

        distancias = {}
        for _, row in manzanas_proj.iterrows():
            dists = equip_pts.distance(row["centroide"])
            distancias[row["cod_manzana"]] = dists.min()
        return pd.Series(distancias, name=self.name)

ise_ind = IndicadorEducacion(manzanas_apartado, educ_gdf)
ise_score = ise_ind.run()
print(f"ISE calculado: {len(ise_score)} manzanas")
print(ise_score.describe().round(4))

manzanas_apartado = manzanas_apartado.merge(
    ise_score.rename("ISE").reset_index().rename(columns={"index": "cod_manzana"}),
    on="cod_manzana", how="left"
)

# Mapa ISE
fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_apartado.to_crs(CRS_COL)
manzanas_proj.plot(
    column="ISE", ax=ax, cmap="RdYlBu", legend=True,
    legend_kwds={"label": "ISE — Accesibilidad a Educación (0=peor, 1=mejor)", "shrink": 0.7},
    edgecolor="white", linewidth=0.3, alpha=0.85, vmin=0, vmax=1
)
if not educ_gdf.empty:
    educ_proj = educ_gdf.to_crs(CRS_COL)
    educ_proj.plot(ax=ax, color="blue", marker="s", markersize=50,
                   label="Establecimientos educativos", zorder=5)
try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass
ax.set_title("Accesibilidad a Educación (ISE) — Apartadó", fontsize=14, fontweight="bold")
ax.legend(loc="lower left")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "ise_apartado.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Mapa comparativo: salud vs educación (subplots 1x2) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
manzanas_proj = manzanas_apartado.to_crs(CRS_COL)

for ax, col, titulo, cmap_, equip, equip_color, marker in [
    (axes[0], "ISAL", "Accesibilidad Salud (ISAL)", "RdYlGn", salud_gdf, "red", "+"),
    (axes[1], "ISE",  "Accesibilidad Educación (ISE)", "RdYlBu", educ_gdf, "blue", "s"),
]:
    manzanas_proj.plot(
        column=col, ax=ax, cmap=cmap_, legend=True,
        legend_kwds={"label": f"{col} (0=peor, 1=mejor)", "shrink": 0.7},
        edgecolor="white", linewidth=0.3, alpha=0.85, vmin=0, vmax=1
    )
    if not equip.empty:
        equip.to_crs(CRS_COL).plot(
            ax=ax, color=equip_color, marker=marker,
            markersize=80, label=col, zorder=5
        )
    try:
        ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
    except Exception:
        pass
    ax.set_title(titulo, fontsize=13, fontweight="bold")
    ax.set_axis_off()

plt.suptitle("Comparativo de Accesibilidad — Apartadó", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "comparativo_isal_ise_apartado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa comparativo guardado.")


In [ ]:
# ── Identificar 'desiertos de servicios' ─────────────────────────────────
# Manzanas con ISAL < 0.2 Y ISE < 0.2: sin acceso mínimo a salud ni educación
UMBRAL = 0.2

desiertos = manzanas_apartado[
    (manzanas_apartado["ISAL"] < UMBRAL) &
    (manzanas_apartado["ISE"] < UMBRAL)
].copy()

print(f"Manzanas 'desierto de servicios' (ISAL<{UMBRAL} y ISE<{UMBRAL}): {len(desiertos)}")
print(f"Porcentaje del total: {100 * len(desiertos) / len(manzanas_apartado):.1f}%")

fig, ax = plt.subplots(figsize=(12, 10))
manzanas_proj = manzanas_apartado.to_crs(CRS_COL)

# Fondo: todas las manzanas en gris
manzanas_proj.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.3, alpha=0.6)

# Resaltar desiertos en rojo
if not desiertos.empty:
    desiertos.to_crs(CRS_COL).plot(
        ax=ax, color="crimson", edgecolor="white", linewidth=0.5, alpha=0.9,
        label=f"Desierto de servicios (n={len(desiertos)})"
    )

# Equipamientos
for equip, color, marker, label in [
    (salud_gdf, "red", "+", "Salud"),
    (educ_gdf, "blue", "s", "Educación")
]:
    if not equip.empty:
        equip.to_crs(CRS_COL).plot(ax=ax, color=color, marker=marker,
                                    markersize=80, label=label, zorder=5)

try:
    ctx.add_basemap(ax, crs=CRS_COL, source=ctx.providers.CartoDB.Positron)
except Exception:
    pass

ax.set_title(f"Desiertos de Servicios — ISAL<{UMBRAL} y ISE<{UMBRAL} — Apartadó",
             fontsize=13, fontweight="bold")
ax.legend(loc="lower left", fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(ROOT / "data" / "outputs" / "desiertos_servicios_apartado.png", dpi=150, bbox_inches="tight")
plt.show()

if not desiertos.empty:
    print("\nManzanas desierto:")
    display(desiertos[["cod_manzana", "ISAL", "ISE"]].sort_values("ISAL").head(10))


## Análisis cualitativo de brechas encontradas

### Salud (ISAL)
- Las manzanas con **ISAL < 0.2** corresponden típicamente a zonas periféricas de Apartadó
  a más de 1.5 km del hospital o centro de salud más cercano.
- El Hospital Antonio Roldán Betancur (II nivel) concentra la oferta hospitalaria,
  generando un gradiente claro de accesibilidad desde el centro.
- **Brecha crítica**: sectores al sur-oriente (barrios de expansión reciente) sin oferta primaria.

### Educación (ISE)
- La oferta educativa oficial tiene mejor distribución territorial que la salud,
  con varias instituciones de primaria descentralizadas.
- Sin embargo, la oferta de **media vocacional** (grados 10-11) es más concentrada,
  generando mayores distancias para adolescentes en zonas periféricas.

### Desiertos de servicios
- Las manzanas con déficit simultáneo en salud y educación merecen atención prioritaria
  en la priorización de inversión pública.
- En municipios más pequeños (Necoclí, San Juan, Arboletes) se espera que la proporción
  de manzanas-desierto sea significativamente mayor — validar en Notebook 06.

### Limitaciones metodológicas
- Se usa distancia euclidiana (vuelo de pájaro), no tiempo de viaje real.
  Fase 2: usar pgRouting + red vial OSM para distancias en red.
- No se pondera por capacidad instalada del equipamiento (camas, matrícula disponible).
- La accesibilidad a servicios de segundo y tercer nivel requiere análisis regional
  que excede el perímetro municipal.
